# Step 1: Business Problem Definition

## Business Question

Which customers are most likely to respond to a commercial campaign, and how should the business prioritize outreach actions?

## Project Goal

The goal of this project is to predict customer campaign response probability and convert the prediction results into practical outreach recommendations.

The project is designed as a commercial analytics decision-support project. Therefore, the focus is not only on model accuracy, but also on business usefulness, customer prioritization, and campaign efficiency.

## Business Use Case

A marketing or commercial team has a limited budget and limited sales capacity. Instead of contacting all customers equally, the business wants to identify which customers are more likely to respond and prioritize them for outreach.

The final model should support:

- Customer ranking
- Campaign targeting
- Outreach prioritization
- Resource allocation
- Response-rate improvement


# Step 2: Real Dataset Selection

## Dataset Used

This project uses the UCI Bank Marketing Dataset.

The dataset is based on direct marketing campaigns conducted by a Portuguese banking institution. The campaigns were mainly based on phone calls, and the business goal was to determine whether a customer would subscribe to a term deposit.

## Target Variable

The target variable is:

**y = whether the customer subscribed to a term deposit**

This project treats `y` as the campaign-response outcome.

- `yes` means the customer subscribed to a term deposit.
- `no` means the customer did not subscribe to a term deposit.

For machine learning, the target will later be converted into a binary variable:

- `1` = positive campaign response
- `0` = negative campaign response

## Business Interpretation

In this project, subscribing to a term deposit is interpreted as a successful customer response to a commercial campaign. Therefore, the model will be used to estimate which customers are most likely to respond positively and should be prioritized for outreach.

In [72]:

import pandas as pd

# Load the UCI Bank Marketing dataset
data_path = "../data/raw/bank-additional-full.csv"

df = pd.read_csv(data_path, sep=";")

# Display basic information
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [73]:
# Check columns
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week',
       'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed', 'y'],
      dtype='object')

In [74]:
# Check target variable distribution
df["y"].value_counts()

y
no     36548
yes     4640
Name: count, dtype: int64

In [75]:
# Check target variable distribution as percentages
df["y"].value_counts(normalize=True) * 100

y
no     88.734583
yes    11.265417
Name: proportion, dtype: float64

In [76]:
# Create binary campaign response target
df["campaign_response"] = df["y"].map({"yes": 1, "no": 0})

df[["y", "campaign_response"]].head()

,y,campaign_response
0,no,0
1,no,0
2,no,0
3,no,0
4,no,0


In [77]:
# Confirm binary target distribution
df["campaign_response"].value_counts()

campaign_response
0    36548
1     4640
Name: count, dtype: int64

In [78]:
# Confirm binary target distribution as percentages
df["campaign_response"].value_counts(normalize=True) * 100

campaign_response
0    88.734583
1    11.265417
Name: proportion, dtype: float64

## Data Leakage Note

The variable `duration` represents the last contact duration in seconds. Since this information is only known after the customer has already been contacted, it should not be used in a pre-campaign targeting model.

For commercial outreach prioritization, the business needs to rank customers before contacting them. Therefore, `duration` will be excluded from the predictive modeling stage to avoid data leakage.

This is important because the UCI documentation defines `duration` as the last contact duration, which means it would not be available before outreach.

# Step 3: Load and Understand the Data

This step loads the UCI Bank Marketing Dataset and performs initial data understanding for commercial campaign analytics.

The analysis focuses on:

- Number of customers
- Campaign response rate
- Numerical and categorical variables
- Missing or unknown values
- Campaign-related variables
- Previous campaign outcomes
- Business questions related to customer response and outreach prioritization

In [79]:
#Cell 2 — Import Libraries
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [80]:
#Cell 3 — Load the Dataset
from pathlib import Path

data_path = Path("../data/raw/bank-additional-full.csv")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {data_path}. "
        "Make sure bank-additional-full.csv is saved inside data/raw/"
    )

df = pd.read_csv(data_path, sep=";")

df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [81]:
#Cell 4 — Basic Dataset Size
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

df.info()

Number of rows: 41188
Number of columns: 21
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  

In [82]:
#Cell 5 — Check Column Names
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

In [83]:
#Cell 6 — Response Rate
response_counts = df["y"].value_counts()
response_rate = df["y"].value_counts(normalize=True) * 100

response_summary = pd.DataFrame({
    "count": response_counts,
    "percentage": response_rate.round(2)
})

response_summary

,count,percentage
y,,
no,36548,88.73
yes,4640,11.27


In [84]:
#Cell 7 — Create Binary Response Variable
df["response_flag"] = df["y"].map({"yes": 1, "no": 0})

df[["y", "response_flag"]].head()

,y,response_flag
0,no,0
1,no,0
2,no,0
3,no,0
4,no,0


In [85]:
#Cell 8 — Overall Campaign Response Rate
overall_response_rate = df["response_flag"].mean() * 100

print(f"Overall response rate: {overall_response_rate:.2f}%")

Overall response rate: 11.27%


Only a small percentage of customers responded positively to the campaign. 
This confirms that customer prioritization is important because contacting all customers equally may be inefficient.

In [86]:
#Cell 9 — Numerical Variables
numerical_vars = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

numerical_vars

['age',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'response_flag']

In [87]:
#Cell 10 — Numerical Summary
df[numerical_vars].describe().T

,count,mean,std,min,25%,50%,75%,max
age,41188.0,40.024060,10.421250,17.000,32.000,38.000,47.000,98.000
duration,41188.0,258.285010,259.279249,0.000,102.000,180.000,319.000,4918.000
campaign,41188.0,2.567593,2.770014,1.000,1.000,2.000,3.000,56.000
pdays,41188.0,962.475454,186.910907,0.000,999.000,999.000,999.000,999.000
previous,41188.0,0.172963,0.494901,0.000,0.000,0.000,0.000,7.000
emp.var.rate,41188.0,0.081886,1.570960,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,41188.0,93.575664,0.578840,92.201,93.075,93.749,93.994,94.767
cons.conf.idx,41188.0,-40.502600,4.628198,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,41188.0,3.621291,1.734447,0.634,1.344,4.857,4.961,5.045
nr.employed,41188.0,5167.035911,72.251528,4963.600,5099.100,5191.000,5228.100,5228.100


In [88]:
#Cell 11 — Categorical Variables
categorical_vars = df.select_dtypes(include=["object"]).columns.tolist()

categorical_vars

['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'poutcome',
 'y']

In [89]:
#Cell 12 — Categorical Summary
categorical_summary = []

for col in categorical_vars:
    categorical_summary.append({
        "variable": col,
        "unique_values": df[col].nunique(),
        "most_common_value": df[col].mode()[0],
        "most_common_count": df[col].value_counts().iloc[0]
    })

categorical_summary_df = pd.DataFrame(categorical_summary)

categorical_summary_df

,variable,unique_values,most_common_value,most_common_count
0,job,12,admin.,10422
1,marital,4,married,24928
2,education,8,university.degree,12168
3,default,3,no,32588
4,housing,3,yes,21576
5,loan,3,no,33950
6,contact,2,cellular,26144
7,month,10,may,13769
8,day_of_week,5,thu,8623
9,poutcome,3,nonexistent,35563


In [90]:
#Cell 13 — Missing Values
missing_values = df.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

Series([], dtype: int64)

In [91]:
#Cell 14 — Unknown Values
unknown_counts = {}

for col in categorical_vars:
    unknown_counts[col] = (df[col] == "unknown").sum()

unknown_summary = pd.DataFrame.from_dict(
    unknown_counts, 
    orient="index", 
    columns=["unknown_count"]
)

unknown_summary["unknown_percentage"] = (
    unknown_summary["unknown_count"] / len(df) * 100
).round(2)

unknown_summary = unknown_summary.sort_values(
    by="unknown_count", 
    ascending=False
)

unknown_summary

,unknown_count,unknown_percentage
default,8597,20.87
education,1731,4.20
housing,990,2.40
loan,990,2.40
job,330,0.80
marital,80,0.19
contact,0,0.00
month,0,0.00
day_of_week,0,0.00
poutcome,0,0.00


Some customer attributes contain "unknown" values. These are not standard missing values, but they still represent incomplete customer information. 
For modeling, these values may be kept as a separate category or handled during preprocessing.

In [92]:
#Cell 15 — Campaign-Related Variables
campaign_vars = [
    "contact",
    "month",
    "day_of_week",
    "duration",
    "campaign",
    "pdays",
    "previous",
    "poutcome"
]

df[campaign_vars].head()

,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome
0,telephone,may,mon,261,1,999,0,nonexistent
1,telephone,may,mon,149,1,999,0,nonexistent
2,telephone,may,mon,226,1,999,0,nonexistent
3,telephone,may,mon,151,1,999,0,nonexistent
4,telephone,may,mon,307,1,999,0,nonexistent


In [93]:
#Cell 16 — Campaign Variable Summary
df[campaign_vars].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
contact,41188,2,cellular,26144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month,41188,10,may,13769,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,41188,5,thu,8623,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,41188.0,NaN,NaN,NaN,258.28501,259.279249,0.0,102.0,180.0,319.0,4918.0
campaign,41188.0,NaN,NaN,NaN,2.567593,2.770014,1.0,1.0,2.0,3.0,56.0
pdays,41188.0,NaN,NaN,NaN,962.475454,186.910907,0.0,999.0,999.0,999.0,999.0
previous,41188.0,NaN,NaN,NaN,0.172963,0.494901,0.0,0.0,0.0,0.0,7.0
poutcome,41188,3,nonexistent,35563,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [94]:
#Business Question 1
#Which Customer Groups Respond More Often?
#Cell 17 — Response Rate by Job
response_by_job = (
    df.groupby("job")["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="response_rate", ascending=False)
)

response_by_job["response_rate"] = (response_by_job["response_rate"] * 100).round(2)

response_by_job

,customer_count,response_rate
job,,
student,875,31.43
retired,1720,25.23
unemployed,1014,14.20
admin.,10422,12.97
management,2924,11.22
unknown,330,11.21
technician,6743,10.83
self-employed,1421,10.49
housemaid,1060,10.00


In [95]:
#Cell 18 — Response Rate by Marital Status
response_by_marital = (
    df.groupby("marital")["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="response_rate", ascending=False)
)

response_by_marital["response_rate"] = (response_by_marital["response_rate"] * 100).round(2)

response_by_marital

,customer_count,response_rate
marital,,
unknown,80,15.00
single,11568,14.00
divorced,4612,10.32
married,24928,10.16


In [96]:
#Cell 19 — Response Rate by Education
response_by_education = (
    df.groupby("education")["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="response_rate", ascending=False)
)

response_by_education["response_rate"] = (response_by_education["response_rate"] * 100).round(2)

response_by_education

,customer_count,response_rate
education,,
illiterate,18,22.22
unknown,1731,14.50
university.degree,12168,13.72
professional.course,5243,11.35
high.school,9515,10.84
basic.4y,4176,10.25
basic.6y,2292,8.20
basic.9y,6045,7.82


In [97]:
#Cell 20 — Create Age Groups
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 30, 40, 50, 60, 100],
    labels=["Under 30", "30-39", "40-49", "50-59", "60+"]
)

response_by_age_group = (
    df.groupby("age_group", observed=False)["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="response_rate", ascending=False)
)

response_by_age_group["response_rate"] = (
    response_by_age_group["response_rate"] * 100
).round(2)

response_by_age_group

,customer_count,response_rate
age_group,,
60+,910,45.49
Under 30,7383,15.22
50-59,6270,10.65
30-39,16385,9.75
40-49,10240,8.17


In [98]:
#Business Question 2
#Which Contact Types Perform Better?
#Cell 21 — Response Rate by Contact Type
response_by_contact = (
    df.groupby("contact")["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="response_rate", ascending=False)
)

response_by_contact["response_rate"] = (
    response_by_contact["response_rate"] * 100
).round(2)

response_by_contact

,customer_count,response_rate
contact,,
cellular,26144,14.74
telephone,15044,5.23


This analysis compares campaign performance across contact channels. 
If one contact type has a meaningfully higher response rate, the business may prioritize that channel in future outreach planning.

In [99]:
#Business Question 3
#Does Prior Campaign Success Matter?
#Cell 22 — Previous Campaign Outcome
response_by_poutcome = (
    df.groupby("poutcome")["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="response_rate", ascending=False)
)

response_by_poutcome["response_rate"] = (
    response_by_poutcome["response_rate"] * 100
).round(2)

response_by_poutcome

,customer_count,response_rate
poutcome,,
success,1373,65.11
failure,4252,14.23
nonexistent,35563,8.83


The previous campaign outcome is important because customers who responded positively before may have a higher probability of responding again.
This can help the business prioritize customers with stronger historical engagement.

In [100]:
#Cell 23 — Previous Number of Contacts
response_by_previous_contacts = (
    df.groupby("previous")["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
    .sort_values(by="previous")
)

response_by_previous_contacts["response_rate"] = (
    response_by_previous_contacts["response_rate"] * 100
).round(2)

response_by_previous_contacts

,customer_count,response_rate
previous,,
0,35563,8.83
1,4561,21.20
2,754,46.42
3,216,59.26
4,70,54.29
5,18,72.22
6,5,60.00
7,1,0.00


In [101]:
#Business Question 4
#Are Too Many Contacts Reducing Response?
#Cell 24 — Current Campaign Contact Frequency
df["campaign_contact_group"] = pd.cut(
    df["campaign"],
    bins=[0, 1, 2, 3, 5, 10, 100],
    labels=[
        "1 contact",
        "2 contacts",
        "3 contacts",
        "4-5 contacts",
        "6-10 contacts",
        "More than 10 contacts"
    ]
)

response_by_campaign_contacts = (
    df.groupby("campaign_contact_group", observed=False)["response_flag"]
    .agg(["count", "mean"])
    .rename(columns={"count": "customer_count", "mean": "response_rate"})
)

response_by_campaign_contacts["response_rate"] = (
    response_by_campaign_contacts["response_rate"] * 100
).round(2)

response_by_campaign_contacts

,customer_count,response_rate
campaign_contact_group,,
1 contact,17642,13.04
2 contacts,10570,11.46
3 contacts,5341,10.75
4-5 contacts,4250,8.68
6-10 contacts,2516,6.32
More than 10 contacts,869,3.11


This analysis checks whether repeated contact attempts are associated with lower customer response.
If response rates decline after multiple contacts, the business should avoid excessive outreach and reduce customer fatigue.

In [102]:
#Cell 25 — Important Commercial Insight Tables
business_tables = {
    "response_by_job": response_by_job,
    "response_by_marital": response_by_marital,
    "response_by_education": response_by_education,
    "response_by_age_group": response_by_age_group,
    "response_by_contact": response_by_contact,
    "response_by_poutcome": response_by_poutcome,
    "response_by_campaign_contacts": response_by_campaign_contacts
}

for name, table in business_tables.items():
    print(f"\n{name}")
    display(table)


response_by_job


,customer_count,response_rate
job,,
student,875,31.43
retired,1720,25.23
unemployed,1014,14.20
admin.,10422,12.97
management,2924,11.22
unknown,330,11.21
technician,6743,10.83
self-employed,1421,10.49
housemaid,1060,10.00



response_by_marital


,customer_count,response_rate
marital,,
unknown,80,15.00
single,11568,14.00
divorced,4612,10.32
married,24928,10.16



response_by_education


...

,customer_count,response_rate
contact,,
cellular,26144,14.74
telephone,15044,5.23



response_by_poutcome


,customer_count,response_rate
poutcome,,
success,1373,65.11
failure,4252,14.23
nonexistent,35563,8.83



response_by_campaign_contacts


,customer_count,response_rate
campaign_contact_group,,
1 contact,17642,13.04
2 contacts,10570,11.46
3 contacts,5341,10.75
4-5 contacts,4250,8.68
6-10 contacts,2516,6.32
More than 10 contacts,869,3.11


#Cell 26 — Step 3 Summary Markdown
## Step 3 Summary: Data Understanding

The dataset contains 41,188 customer campaign records and 21 original variables. The target variable is `y`, which indicates whether a customer subscribed to a term deposit after the campaign.

The overall campaign response rate is relatively low, meaning that most customers did not respond positively. This supports the commercial need for customer prioritization rather than equal outreach to all customers.

The dataset includes both numerical and categorical variables. Numerical variables include customer age, number of campaign contacts, previous contacts, and macroeconomic indicators. Categorical variables include job type, marital status, education, loan status, contact type, month, day of week, and previous campaign outcome.

The dataset does not contain standard missing values, but several categorical variables contain `"unknown"` values. These values should be handled carefully during preprocessing.

Initial business analysis suggests that response rates vary across customer groups, contact types, prior campaign outcomes, and number of campaign contacts. These patterns are useful for building a commercial analytics model that ranks customers by predicted response probability and supports outreach prioritization.